In [1]:
import spacy as sp
import pandas as pd
import numpy as np
import json
import os
from glob import glob
import re
import nltk

In [2]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt to /Users/arocha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /Users/arocha/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [3]:
article_pat = r'\n(?:ARTICLE|Article|Section)\s+(?:[IVX0-9]+)'

In [4]:
with open('constitutions.json', 'r') as file:
    constitutions = json.load(file)

In [5]:
corpus_data = []

for key, text in constitutions.items():
    # Split text into articles
    articles = re.split(article_pat, text)
    
    for art_idx, art_text in enumerate(articles):
        # Break article into sentences
        sentences = nltk.sent_tokenize(art_text)
        
        for sent_idx, sent_text in enumerate(sentences):
            # Break sentence into tokens
            tokens = nltk.word_tokenize(sent_text)
            
            # Get POS tags (NLTK returns list of tuples: [('word', 'TAG')])
            pos_tags = nltk.pos_tag(tokens, tagset='universal')
            
            for token_idx, (token_str, pos) in enumerate(pos_tags):
                corpus_data.append({
                    'country_id': key,
                    'article_id': art_idx,
                    'sent_id': sent_idx,
                    'token_id': token_idx,
                    'token_str': token_str,
                    'term_str': token_str.lower().replace(r'[\W_]+', ''), # Clean term
                    'pos': pos,
                    'pos_group': pos # Universal tagset gives you groups like 'NOUN', 'VERB'
                })

CORPUS = pd.DataFrame(corpus_data)
CORPUS.set_index(['country_id', 'article_id', 'sent_id', 'token_id'], inplace=True)

In [6]:
CORPUS.head()

token_str     term_str   pos  \
country_id       article_id sent_id token_id                                   
Afghanistan_2004 0          0       0         Afghanistan  afghanistan   ADJ   
                                    1                2004         2004   NUM   
                                    2            Preamble     preamble   ADJ   
                                    3               Share        share  NOUN   
                                    4                  In           in   ADP   

                                             pos_group  
country_id       article_id sent_id token_id            
Afghanistan_2004 0          0       0              ADJ  
                                    1              NUM  
                                    2              ADJ  
                                    3             NOUN  
                                    4              ADP

In [7]:
check_report = {}

for key, text in constitutions.items():
    found = re.findall(article_pat, text)
    check_report[key] = {
        'article_count': len(found),
        'char_len': len(text),
        'avg_chars_per_article': len(text) / len(found) if len(found) > 0 else len(text)
    }

# Convert to DataFrame for easy viewing
df_check = pd.DataFrame(check_report).T

In [9]:
df_check[df_check['article_count'] == 0]

,article_count,char_len,avg_chars_per_article
Antigua_and_Barbuda_1981,0.0,214275.0,214275.0
Australia_1985,0.0,81224.0,81224.0
Bahamas_2002,0.0,227497.0,227497.0
Bangladesh_2011,0.0,158974.0,158974.0
Barbados_2007,0.0,199629.0,199629.0
Belize_2001,0.0,235373.0,235373.0
Botswana_2002,0.0,180775.0,180775.0
Brazil_2014,0.0,437141.0,437141.0
Brunei_1984,0.0,82463.0,82463.0
Denmark_1953,0.0,36816.0,36816.0


In [10]:
import re
import pandas as pd
import nltk

# Pattern 1: Section numbers (e.g., "1.", "Section 1", "Article 5")
section_pat = r'\n(?:Section|Article|ARTICLE)?\s*([0-9]+)[\.\s]'

# Pattern 2: Subsection letters (e.g., "(a)", "b)", "c.")
# This looks for a letter at the start of a line or after a bit of whitespace
sub_pat = r'\n\s*[\(\[]?([a-z])[\)\.\s]'

corpus_data = []

for key, text in constitutions.items():
    # Tier 1: Split into Sections (Numbers)
    sections = re.split(section_pat, text)
    
    for sec_idx, sec_text in enumerate(sections):
        # Tier 2: Split into Subsections (Letters)
        subsections = re.split(sub_pat, sec_text)
        
        for sub_idx, sub_text in enumerate(subsections):
            # Tier 3: Sentences
            sentences = nltk.sent_tokenize(sub_text)
            
            for sent_idx, sent_text in enumerate(sentences):
                # Tier 4: Tokens & POS Tagging
                tokens = nltk.word_tokenize(sent_text)
                pos_tags = nltk.pos_tag(tokens, tagset='universal')
                
                for tok_idx, (token_str, pos) in enumerate(pos_tags):
                    # Basic cleaning for term_str (Requirement F4 prep)
                    term_str = token_str.lower().strip().replace('.', '')
                    
                    if not term_str: continue # Skip empty strings
                    
                    corpus_data.append({
                        'country_id': key,
                        'section_id': sec_idx,
                        'sub_id': sub_idx,
                        'sent_id': sent_idx,
                        'token_id': tok_idx,
                        'token_str': token_str,
                        'term_str': term_str,
                        'pos': pos,
                        'pos_group': pos
                    })

CORPUS = pd.DataFrame(corpus_data)
CORPUS.set_index(['country_id', 'section_id', 'sub_id', 'sent_id', 'token_id'], inplace=True)

In [17]:
CORPUS.head()

token_str     term_str  \
country_id       section_id sub_id sent_id token_id                             
Afghanistan_2004 0          0      0       0         Afghanistan  afghanistan   
                                           1                2004         2004   
                                           2            Preamble     preamble   
                                           3               Share        share   
                                           4                  In           in   

                                                      pos pos_group  
country_id       section_id sub_id sent_id token_id                  
Afghanistan_2004 0          0      0       0          ADJ       ADJ  
                                           1          NUM       NUM  
                                           2          ADJ       ADJ  
                                           3         NOUN      NOUN  
                                           4          ADP       ADP

In [18]:
# Calculate the "Branching Factor" for each country
# This shows how many units are inside the unit above it
idx_check = CORPUS.groupby('country_id').apply(lambda x: pd.Series({
    'total_tokens': len(x),
    'n_sections': x.index.get_level_values('section_id').nunique(),
    'n_subs': x.index.get_level_values('sub_id').nunique(),
    'avg_sents_per_sub': x.index.get_level_values('sent_id').nunique() / x.index.get_level_values('sub_id').nunique()
}))

print("--- OHCO Branching Factor ---")
print(idx_check)

--- OHCO Branching Factor ---
                  total_tokens  n_sections  n_subs  avg_sents_per_sub
country_id                                                           
Afghanistan_2004       11122.0       483.0     1.0           7.000000
Albania_2008           14580.0      1185.0    25.0           0.240000
Algeria_2008           11138.0       495.0     1.0          17.000000
Andorra_1993            9115.0       547.0    25.0           0.200000
Angola_2010            28972.0      1617.0    47.0           0.234043
...                        ...         ...     ...                ...
Vanuatu_1983            8896.0       497.0    23.0           0.217391
Venezuela_2009         39062.0      1285.0     1.0          30.000000
Yemen_2001             11301.0       405.0    21.0           0.571429
Zambia_2009            32534.0      1147.0    61.0           0.065574
Zimbabwe_2013          60544.0      2569.0    41.0           0.707317

[192 rows x 4 columns]


In [24]:
idx_check[idx_check['n_sections']==1]

,total_tokens,n_sections,n_subs,avg_sents_per_sub
country_id,,,,
Brazil_2014,73677.0,1.0,573.0,0.598604
Italy_2012,12231.0,1.0,35.0,10.400000


In [25]:
flat_docs = idx_check[(idx_check.n_sections == 1) | (idx_check.n_subs == 1)]

if not flat_docs.empty:
    print("⚠️ Potential Parsing Failures (Flat Hierarchy):")
    print(flat_docs.index.tolist())
else:
    print("✅ All documents show hierarchical depth.")

⚠️ Potential Parsing Failures (Flat Hierarchy):
['Afghanistan_2004', 'Algeria_2008', 'Argentina_1994', 'Azerbaijan_2009', 'Benin_1990', 'Bolivia_2009', 'Brazil_2014', 'Bulgaria_2007', 'Burkina_Faso_2012', 'Cambodia_1999', 'Central_African_Republic_2010', 'China_2004', 'Comoros_2009', 'Cote_DIvoire_2009', 'Croatia_2001', 'Denmark_1953', 'Djibouti_2010', 'Egypt_2014', 'El_Salvador_2003', 'Estonia_2003', 'Finland_2011', 'France_2008', 'Gabon_1997', 'Georgia_2004', 'Guinea_2010', 'Iceland_1999', 'Indonesia_2002', 'Iraq_2005', 'Italy_2012', 'Japan_1946', 'Kazakhstan_1998', 'Kosovo_2008', 'Kyrgyz_Republic_2010', 'Laos_2003', 'Latvia_2007', 'Libya_2011', 'Lithuania_2006', 'Luxembourg_2009', 'Macedonia_2011', 'Mauritania_2012', 'Mongolia_2001', 'Montenegro_2007', 'Niger_2010', 'Oman_2011', 'Paraguay_2011', 'Peoples_Republic_of_Korea_1998', 'Poland_1997', 'Qatar_2003', 'Republic_of_Korea_1987', 'Senegal_2009', 'Serbia_2006', 'Socialist_Republic_of_Vietnam_2001', 'Taiwan_2005', 'Tajikistan_2003'